# 15. Procedures: a dark, then measurements

A **procedure** is what one does with a bench, step after step: close the shutter, take frames, average them
into a dark, open the shutter, take frames and subtract the dark. It keeps **named results** from one step to
the next and runs the same way in simulation and in operation (ADR-0024). It is data, not code: in the web
interface it is drawn as blocks (tab *Procedures*). The preset `dark_then_measure.lumos` holds a laser beam on a
camera behind a shutter, and this procedure.

In [ ]:
# On Google Colab only: install LUMOS and fetch the presets used by the examples.
import sys

if "google.colab" in sys.modules:
    %pip install -q "lumos-maxima[catalogs] @ git+https://github.com/VForiel/LUMOS"
    !git clone -q --depth 1 https://github.com/VForiel/LUMOS /content/LUMOS
    !cp -r /content/LUMOS/examples/presets /content/LUMOS/examples/catalog_cache .

In [ ]:
import json

import numpy as np
from astropy import units as u

import lumos
from lumos import plotting
from lumos.plotting import style

style.apply()
ctx = lumos.io.load("presets/dark_then_measure.lumos")
procedure = ctx.procedures["dark then measure"]
print(json.dumps(procedure.to_json()["steps"][:3], indent=1))

Before running, `problems` checks the procedure against its context (elements, ports, actions, functions,
names used before they are defined); an empty list means it can run.

In [ ]:
print(procedure.problems(ctx))
result = ctx.run_procedure(procedure)
print(sorted(result.results), "shown:", result.shown)
print("time on the clock:", ctx.elapsed)

The dark is the mean of 20 frames taken shutter closed: about `D t / g + bias` = 200 ADU per pixel, with a
noise divided by √20. The science frame, the mean of 10 frames minus the dark, keeps only the light of the
laser.

In [ ]:
dark = result["dark"].data.to_value(u.adu)
print(
    f"dark: mean {dark.mean():.1f} ADU, pixel noise {dark.std():.2f} ADU "
    f"(expected {np.sqrt(200 * 0.5 + 5**2) / np.sqrt(20):.2f})"
)
fig, axes = style.figure("double", ncols=2, aspect=0.45)
plotting.plot_data(result["dark"], ax=axes[0])
axes[0].set_title("dark")
plotting.plot_data(result["science"], ax=axes[1])
axes[1].set_title("science = images - dark");

The figures and the data are kept in an archive of the run, with the context and its journal:

In [ ]:
from lumos.results import new_archive

archive = new_archive(ctx, "dark-then-measure")
archive.save_figure(
    fig, "dark_and_science", data={"dark": result["dark"].data, "science": result["science"].data}
)
archive.path.name

The procedure exports to an equivalent Python script (blocks to script, not the reverse):

In [ ]:
print(procedure.to_python())